# 03 · Rainfall EDA — NOAA MRMS radar rainfall (QPE) on AWS
**Project:** Real-Time Flood & Flash-Flood Forecasting and Alerting using Apache Kafka (AI372IA, Phase I)

**Dataset:** NOAA Multi-Radar/Multi-Sensor (MRMS) quantitative precipitation estimates, public S3 bucket `noaa-mrms-pds` (no sign-in). Files are gzipped GRIB2 grids over the continental US (~1 km).

**This notebook:**
1. explores what MRMS products and files exist (names, sizes, update cadence);
2. measures how soon after the hour an hourly rainfall file is published (feed delay for our rainfall producer);
3. decodes one hourly grid and explores its values (missing codes, distribution, spatial pattern around a gauge);
4. extracts rainfall at a gauge for a past event and compares it with the river stage.

**How to run:** Colab or Kaggle (Internet on). The first cell installs `cfgrib`/`eccodes` for GRIB2 decoding.

In [ ]:
!pip install -q cfgrib eccodes xarray

## 1. Setup and configuration

In [ ]:
import json, os, time, gzip, io, math
from datetime import datetime, timedelta, timezone
import numpy as np
import pandas as pd
import requests

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
UA = {"User-Agent": "rvce-flood-eda-notebook"}

def get_json(url, params=None, retries=3):
    """GET with retries; raises on final failure."""
    last = None
    for i in range(retries):
        try:
            r = requests.get(url, params=params, headers=UA, timeout=60)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last = e
            time.sleep(2 * (i + 1))
    raise RuntimeError(f"GET failed: {url} {params}: {last}")

def parse_iv(doc):
    """USGS Instantaneous Values JSON -> tidy DataFrame (one row per reading)."""
    rows, sites = [], {}
    for ts in doc["value"]["timeSeries"]:
        src, var = ts["sourceInfo"], ts["variable"]
        site = src["siteCode"][0]["value"]
        geo = src.get("geoLocation", {}).get("geogLocation", {})
        sites[site] = {"site_no": site, "site_name": src.get("siteName"),
                       "lat": geo.get("latitude"), "lon": geo.get("longitude")}
        param = var["variableCode"][0]["value"]
        nodata = var.get("noDataValue")
        for i, block in enumerate(ts.get("values", [])):
            meth = (block.get("method") or [{}])[0]
            series = str(meth.get("methodID", i))
            mdesc = meth.get("methodDescription", "")
            for pt in block.get("value", []):
                try:
                    v = float(pt["value"])
                except (TypeError, ValueError):
                    continue
                rows.append((site, param, series, mdesc, pt["dateTime"], v,
                             ",".join(pt.get("qualifiers") or []), v == nodata))
    df = pd.DataFrame(rows, columns=["site_no", "parameter", "series", "method", "dateTime",
                                     "value", "qualifiers", "is_nodata"])
    if len(df):
        df["datetime_utc"] = pd.to_datetime(df["dateTime"], utc=True, format="ISO8601")
        df = df.drop(columns="dateTime")
    return df, pd.DataFrame(sites.values())

def fetch_iv(sites, params, start, end, chunk_days=120, cache=True):
    """Fetch USGS IV data in chunks (the service limits request sizes)."""
    key = f"iv_{'-'.join(sites)}_{'-'.join(params)}_{start}_{end}.csv.gz"
    path = os.path.join(DATA_DIR, key)
    meta_path = path.replace(".csv.gz", "_sites.csv")
    if cache and os.path.exists(path) and os.path.exists(meta_path):
        df = pd.read_csv(path, dtype={"site_no": str, "parameter": str, "series": str})
        df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], utc=True, format="ISO8601")
        return df, pd.read_csv(meta_path, dtype={"site_no": str})
    frames, metas = [], []
    t0, t1 = pd.Timestamp(start), pd.Timestamp(end)
    cur = t0
    while cur < t1:
        nxt = min(cur + pd.Timedelta(days=chunk_days), t1)
        doc = get_json("https://waterservices.usgs.gov/nwis/iv/",
                       {"format": "json", "sites": ",".join(sites), "parameterCd": ",".join(params),
                        "startDT": cur.strftime("%Y-%m-%d"), "endDT": nxt.strftime("%Y-%m-%d"),
                        "siteStatus": "all"})
        d, m = parse_iv(doc)
        frames.append(d); metas.append(m)
        print(f"  {cur.date()} .. {nxt.date()}: {len(d):,} readings")
        cur = nxt + pd.Timedelta(days=1)
    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    meta = pd.concat(metas).drop_duplicates("site_no") if metas else pd.DataFrame()
    df = df.drop_duplicates(["site_no", "parameter", "series", "datetime_utc"])
    if cache:
        df.to_csv(path, index=False); meta.to_csv(meta_path, index=False)
    return df, meta

def primary_series(df, site, param):
    """The series with the most readings is treated as the primary sensor."""
    d = df[(df.site_no == site) & (df.parameter == param) & (~df.is_nodata)]
    if d.empty:
        return pd.Series(dtype=float)
    s = d.series.value_counts().idxmax()
    out = d[d.series == s].set_index("datetime_utc")["value"].sort_index()
    return out[~out.index.duplicated()]

In [ ]:
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import seaborn as sns
import xarray as xr
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 4)

BUCKET = "https://noaa-mrms-pds.s3.amazonaws.com"
PRODUCT = "MultiSensor_QPE_01H_Pass2_00.00"     # hourly multi-sensor QPE (gauge-corrected pass)
# ---- CONFIG: a gauge and a past event (e.g. a crossing time from notebook 02's events table) ----
SITE        = "09510200"
EVENT_UTC   = "2024-07-15T00:00"    # replace with a real event time from notebook 02
HOURS_BEFORE, HOURS_AFTER = 24, 12
BOX_DEG     = 0.5                   # half-width of the map window around the gauge

## 2. What is in the bucket?

In [ ]:
def s3_list(prefix, delimiter=None, max_keys=1000, token=None):
    """Anonymous S3 ListObjectsV2 over HTTPS. Returns (keys, prefixes, next_token)."""
    p = {"list-type": "2", "prefix": prefix, "max-keys": str(max_keys)}
    if delimiter: p["delimiter"] = delimiter
    if token: p["continuation-token"] = token
    r = requests.get(BUCKET + "/", params=p, headers=UA, timeout=60); r.raise_for_status()
    ns = {"s": "http://s3.amazonaws.com/doc/2006-03-01/"}
    root = ET.fromstring(r.content)
    keys = [{"key": c.find("s:Key", ns).text, "size": int(c.find("s:Size", ns).text),
             "last_modified": pd.Timestamp(c.find("s:LastModified", ns).text)} for c in root.findall("s:Contents", ns)]
    prefixes = [c.find("s:Prefix", ns).text for c in root.findall("s:CommonPrefixes", ns)]
    nt = root.find("s:NextContinuationToken", ns)
    return keys, prefixes, (nt.text if nt is not None else None)

_, products, _ = s3_list("CONUS/", delimiter="/")
qpe = [p for p in products if "QPE" in p]
print(f"{len(products)} CONUS products; {len(qpe)} rainfall (QPE) products, e.g.:")
for p in qpe[:25]: print("  ", p)

In [ ]:
def valid_time(key):
    """MRMS file names end with _YYYYMMDD-HHMMSS.grib2.gz"""
    stamp = key.rsplit("_", 1)[1].split(".")[0]
    return pd.Timestamp(datetime.strptime(stamp, "%Y%m%d-%H%M%S"), tz="UTC")

def day_files(product, day):
    keys, _, tok = s3_list(f"CONUS/{product}/{day:%Y%m%d}/")
    while tok:
        more, _, tok = s3_list(f"CONUS/{product}/{day:%Y%m%d}/", token=tok); keys += more
    d = pd.DataFrame(keys)
    if len(d): d["valid_time"] = d.key.map(valid_time)
    return d

yesterday = pd.Timestamp.now(tz="UTC").normalize() - pd.Timedelta(days=1)
files = day_files(PRODUCT, yesterday)
print(f"{PRODUCT} on {yesterday.date()}: {len(files)} files, size {files['size'].min():,}–{files['size'].max():,} bytes")
files.head()

## 3. Publication delay (how long after the hour does the file appear?)

In [ ]:
files["delay_min"] = (files.last_modified - files.valid_time).dt.total_seconds() / 60
display(files.delay_min.describe().round(1))
sns.histplot(files.delay_min, bins=30); plt.xlabel("minutes from valid time to upload")
plt.title(f"{PRODUCT}: publication delay on {yesterday.date()}"); plt.show()
print("Note: archive files re-uploaded later have meaningless LastModified; use recent days for this check.")

In [ ]:
# Compare cadence and delay of a few rainfall products (same day)
rows = []
for prod in [p.split("/")[1] for p in qpe if any(k in p for k in ["QPE_01H", "RadarOnly_QPE_01H", "RadarOnly_QPE_15M"])][:6]:
    try:
        f = day_files(prod, yesterday)
    except Exception as e:
        print(prod, e); continue
    if f.empty: continue
    f["delay_min"] = (f.last_modified - f.valid_time).dt.total_seconds() / 60
    step = f.valid_time.sort_values().diff().dt.total_seconds().median() / 60
    rows.append({"product": prod, "files_per_day": len(f), "cadence_min": step,
                 "median_delay_min": f.delay_min.median(), "median_size_kb": f["size"].median() / 1024})
pd.DataFrame(rows).round(1)

## 4. Decode one hourly grid

In [ ]:
def load_grid(key):
    local = os.path.join(DATA_DIR, os.path.basename(key).replace(".gz", ""))
    if not os.path.exists(local):
        r = requests.get(f"{BUCKET}/{key}", headers=UA, timeout=120); r.raise_for_status()
        with open(local, "wb") as f: f.write(gzip.decompress(r.content))
    ds = xr.open_dataset(local, engine="cfgrib", backend_kwargs={"indexpath": ""})
    var = list(ds.data_vars)[0]
    da = ds[var]
    lon = da.longitude
    if float(lon.max()) > 180:                     # MRMS grids use 0..360 longitudes
        da = da.assign_coords(longitude=(((lon + 180) % 360) - 180))
    return da

key = files.sort_values("valid_time").key.iloc[len(files) // 2]
grid = load_grid(key)
print(key); print(grid)
vals = grid.values.ravel()
print("grid shape:", grid.shape, "| unique negative codes:", np.unique(vals[vals < 0])[:10])

In [ ]:
v = vals[np.isfinite(vals)]
print(f"cells: {v.size:,}  negative (missing/no-coverage codes): {(v < 0).mean() * 100:.1f}%  "
      f"zero: {(v == 0).mean() * 100:.1f}%  rain > 0: {(v > 0).mean() * 100:.2f}%")
pos = v[v > 0]
if pos.size:
    print(pd.Series(pos).describe(percentiles=[.5, .9, .99]).round(2))
    sns.histplot(np.log10(pos), bins=60); plt.xlabel("log10 hourly rain (mm), wet cells only")
    plt.title("Distribution of hourly rainfall over CONUS (one hour)"); plt.show()

## 5. Rainfall around a gauge during a past event

In [ ]:
meta_doc = get_json("https://waterservices.usgs.gov/nwis/iv/",
                    {"format": "json", "sites": SITE, "parameterCd": "00065", "period": "PT2H"})
_, meta = parse_iv(meta_doc)
glat, glon = float(meta.lat.iloc[0]), float(meta.lon.iloc[0])
print(f"{SITE}: {meta.site_name.iloc[0]} at {glat:.4f}, {glon:.4f}")

t_event = pd.Timestamp(EVENT_UTC, tz="UTC")
hours = pd.date_range(t_event - pd.Timedelta(hours=HOURS_BEFORE), t_event + pd.Timedelta(hours=HOURS_AFTER), freq="h")
avail = pd.concat([day_files(PRODUCT, d) for d in sorted({h.normalize() for h in hours})], ignore_index=True)
avail = avail[avail.valid_time.isin(hours)].sort_values("valid_time")
print(f"{len(avail)} of {len(hours)} hourly files available")

def window(da):
    lat_sl = slice(glat + BOX_DEG, glat - BOX_DEG) if float(da.latitude[0]) > float(da.latitude[-1]) else slice(glat - BOX_DEG, glat + BOX_DEG)
    return da.sel(latitude=lat_sl, longitude=slice(glon - BOX_DEG, glon + BOX_DEG))

point, total = [], None
for _, f in avail.iterrows():
    w = window(load_grid(f.key)).where(lambda x: x >= 0)
    point.append({"valid_time": f.valid_time,
                  "rain_at_gauge_mm": float(w.sel(latitude=glat, longitude=glon, method="nearest")),
                  "box_mean_mm": float(w.mean()), "box_max_mm": float(w.max())})
    total = w if total is None else total + w
rain = pd.DataFrame(point).set_index("valid_time")
rain

In [ ]:
if total is not None:
    total.plot(cmap="Blues", figsize=(7, 6), cbar_kwargs={"label": "total rain (mm)"})
    plt.scatter([glon], [glat], color="red", marker="^", s=80, label=f"gauge {SITE}"); plt.legend()
    plt.title(f"MRMS total rainfall, {hours[0]:%Y-%m-%d %H}h – {hours[-1]:%Y-%m-%d %H}h UTC"); plt.show()

In [ ]:
st, _ = fetch_iv([SITE], ["00065"], (hours[0] - pd.Timedelta(days=1)).date().isoformat(),
                 (hours[-1] + pd.Timedelta(days=1)).date().isoformat(), chunk_days=30)
stage = primary_series(st, SITE, "00065")[hours[0]:hours[-1]]
fig, ax1 = plt.subplots(figsize=(12, 4)); ax2 = ax1.twinx()
ax1.bar(rain.index, rain.box_mean_mm, width=1 / 24, alpha=.5, label="box-mean rain (mm/h)")
ax1.plot(rain.index, rain.rain_at_gauge_mm, "b.", label="rain at gauge cell")
ax2.plot(stage.index, stage.values, "r-", label="stage (ft)")
ax1.set_ylabel("rain (mm/h)"); ax2.set_ylabel("stage (ft)"); fig.legend(loc="upper left")
plt.title(f"{SITE}: MRMS rainfall vs river stage"); plt.show()
h = stage.resample("h").mean()
j = pd.concat([rain.box_mean_mm, h.diff().rename("rise")], axis=1).dropna()
if len(j) > 6:
    cc = {L: j.box_mean_mm.corr(j.rise.shift(-L)) for L in range(0, 13)}
    print("rain -> rise cross-correlation by lag (h):", {k: round(float(v), 2) for k, v in cc.items() if pd.notna(v)})

## 6. Takeaways to record in the project log
* product chosen for the rainfall producer and its cadence / file size;
* typical publication delay (adds to the gauge feed delay in the latency budget);
* missing-value codes seen in the grid (negative values are masked above);
* lag between rainfall and the stage rise at the gauge — this is the extra warning time rainfall can buy over gauge-only forecasting.